# Pain check: LLM labels (Kaggle, two runs)

Spec: `docs/superpowers/specs/2026-11-26-llm-pain-labels-design.md`. Numbers only leave Kaggle, never text.

**Both runs:** Settings → Accelerator → **GPU T4 x2**; Internet **on**; Add Input → **Shifaa Arabic Medical Consultations**
(by ahmedseleem); notebook **private**; then **Save Version → Save & Run All**.

- **Run 1** (`RUN = 1`): Qwen2.5-7B-Instruct reads each title against the codebook → `llm_scores.csv`.
- **Run 2** (`RUN = 2`): Add Input → *Notebook Output* → this notebook's run 1; set `THRESHOLD` to the number from
  `llm_gate.py` on the laptop. CAMeLBERT-DA learns the LLM's labels → `teacher_scores_llm.csv`.
  Only run it if the gate passed.

In [ ]:
RUN = 1            # 1 = LLM labels, 2 = teacher on LLM labels
THRESHOLD = None   # run 2 only: the number llm_gate.py printed
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/eyadamgad25/qawam.git", "/tmp/qawam"], check=True)
sys.path.insert(0, "/tmp/qawam/ml/pain")
hits = sorted(Path("/kaggle/input").rglob("Bone_Diseases*.csv"))
assert hits, "Shifaa is not attached: Add Input -> search 'Shifaa Arabic Medical Consultations'"
source = hits[0].parent
print("Shifaa CSVs in", source)

In [ ]:
if RUN == 1:
    import llm_label
    assert llm_label.main([str(source), "/kaggle/working"]) == 0
else:
    assert THRESHOLD is not None, "Set THRESHOLD to the number from llm_gate.py"
    scores = sorted(Path("/kaggle/input").rglob("llm_scores.csv"))
    assert scores, "Attach run 1's output: Add Input -> Notebook Output -> this notebook"
    import teacher_train
    assert teacher_train.main([str(source), "/kaggle/working", "--llm", str(scores[0]), str(THRESHOLD)]) == 0